In [47]:
import os
import numpy as np
import pandas as pd
import hashlib

In [ ]:
#################################
# 1. Load address
#################################
Base_dir = "C:/Users/hyeon/psh/ids/9) Car-Hacking Dataset"

attack_dir = {
    "Dos" : 1,
    "Fuzzing" :2 ,
    "Replay" : 3,
    "Spoofing" : 4
}


### 공격 파일 수집 ###
attack_files = []

for folder, attack_id in attack_dir.items():
    folder_path = os.path.join(Base_dir,folder)

    for fname in os.listdir(folder_path):
        if fname.endswith(".log"):
            attack_files.append({
                "path" : os.path.join(folder_path, fname),
                "attack_id": attack_id
            })

all_files = attack_files

In [52]:
import pandas as pd

def process_log_file(path,attack_id):
    df = pd.read_csv(
        path,
        sep=r",",
        header=None
    )

    df.columns = [
        "timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(df.shape[1] - 4)] + ["Label"]

    df["timestamp"] = df["timestamp"].astype(float)

    def to_payload(row):
        dlc = int(row["DLC"])
        vals = row[[f"b{i}" for i in range(dlc)]].tolist()
        vals = [0 if pd.isna(v) else int(str(v), 16) for v in vals]   # hex byte -> int
        vals = vals + [0] * (8 - len(vals))
        return " ".join(f"{v:02x}" for v in vals)

    df["Payload"] = df.apply(to_payload, axis=1)

    df["CAN_ID"] = "0x" + df["CAN_ID"].astype(str)
    

    # ## ======== hashing ID ========== ##

    # def Hash_ID(canid, M=256):
    #     hashing_id = hashlib.sha256(canid.encode())
    #     hex_id = hashing_id.hexdigest()[:4]
    #     return hex_id

    # s = df["CAN_ID"].astype(str).apply(Hash_ID)
    # hash_df = pd.DataFrame({"Hash_bucket_ID": s.to_numpy()})
    # df = pd.concat([df.reset_index(drop=True),hash_df.reset_index(drop=True)], axis=1)


    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0})

    return df

df = process_log_file("C:/Users/hyeon/psh/ids/9) Car-Hacking Dataset/Dos/DoS_dataset.csv",2)


In [54]:
nan_rows = df[df["Label"].isna()]
print(nan_rows)

            timestamp  CAN_ID  DLC  b0  b1 b2   b3   b4   b5   b6   b7 Label  \
36       1.478198e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
135      1.478198e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
227      1.478198e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
320      1.478198e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
412      1.478198e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
...               ...     ...  ...  ..  .. ..  ...  ...  ...  ...  ...   ...   
3665351  1.478201e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
3665443  1.478201e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
3665536  1.478201e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
3665628  1.478201e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   
3665721  1.478201e+09  0x05f0    2  01  00  R  NaN  NaN  NaN  NaN  NaN   NaN   

                         Payload  Label

In [ ]:
#################################
# 3. 64 X 64 gird Encoding
#################################

def hex_to_onehot(hex):
    idx = int(hex,16)
    v= np.zeros(16,dtype=np.float32)
    v[idx] = 1.0
    return v

def grid_encoding(df):
    one_hot = []

    for i in range(len(df)):
        h = df["Hash_bucket_ID"].iloc[i]
        h = h.lower()

        a = hex_to_onehot(h[0])
        b = hex_to_onehot(h[1])
        c = hex_to_onehot(h[2])
        d = hex_to_onehot(h[3])

        one_hot16 = np.concatenate([a,b,c,d] , axis=0)
        one_hot.append(one_hot16)
    
    return np.stack(one_hot,axis=0)


In [ ]:
#################################
# 4. Slide Window
#################################

def Sliding_window(x, win_size=64, stride=32):
    windows = []
    for start in range(0, len(x)-win_size+1 , stride):
        end = start + win_size
        windows.append(x[start:end])

    return np.stack(windows, axis=0)


def Labeling(df, win_size=64,stride=32, attack_threshold=5):
    labels = []
    y = df["Attack Labeling"].to_numpy()
    attack_label = df["Attack Labeling"].max()

    for start in range(0, len(y)-win_size+1, stride):
        end = start + win_size

        win = y[start:end]
        attack_cn = np.count_nonzero(win != 0)
        if attack_cn >= attack_threshold:
            labels.append(attack_label)

        else:
            labels.append(0)

    return np.array(labels)





In [ ]:
#################################
# 5. main
#################################

all_x, all_y = [],[]

for i in all_files:
    file_path = i["path"]
    attack_id = i["attack_id"]

    df = proccess_log_file(file_path, attack_id)

    x_packet = grid_encoding(df)
    
    x_win = Sliding_window(x_packet)

    y_win = Labeling(df)

    ## x,y shape check
    if x_win.shape[0] != y_win.shape[0]:
        print(f"Window num and Label mismatch: {file_path}")

    all_x.append(x_win)
    all_y.append(y_win)

all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

In [ ]:
#################################
# 5. Save as npy
#################################

save_path = "mirgu_window_dataset.pt"

torch.save(
    {
        "X": torch.tensor(all_X, dtype=torch.float32),
        "y": torch.tensor(all_y, dtype=torch.long)
    },
    save_path
)

print(f" Saved dataset to {save_path}")
print("X shape:", all_X.shape)
print("y shape:", all_y.shape)